# Sentiment Analysis Model Training (GPU)
This notebook trains a DistilBERT model for 3-class sentiment analysis (positive/negative/neutral) using SST-2 dataset.

**Steps:**
1. Install dependencies
2. Train the model on GPU (~2-3 minutes)
3. Download the trained model
4. Copy it to your local project

> **Make sure to select GPU runtime:** Runtime > Change runtime type > T4 GPU

## Step 1: Install Dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn torch accelerate

## Step 2: Configuration

In [ ]:
import os
import torch
import numpy as np
from datasets import load_dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import precision_recall_fscore_support

# Config
MODEL_NAME = "distilbert-base-uncased"
NUM_LABELS = 3
LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}
MAX_SEQ_LENGTH = 128
EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
OUTPUT_DIR = "./trained_model"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 3: Load & Preprocess Dataset

In [ ]:
# Load SST-2 from GLUE benchmark
dataset = load_dataset("glue", "sst2")

# Cast label to plain int to avoid ClassLabel conflict
for split in dataset:
    dataset[split] = dataset[split].cast_column("label", Value("int64"))

# Remap: SST-2 binary (0=neg, 1=pos) -> our 3-class (0=neg, 2=pos)
def remap_labels(example):
    example["label"] = 0 if example["label"] == 0 else 2
    return example

dataset = dataset.map(remap_labels)
dataset = dataset.rename_column("sentence", "text")

print(f"Train: {len(dataset['train'])} samples")
print(f"Validation: {len(dataset['validation'])} samples")

In [ ]:
# Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

tokenized = dataset.map(tokenize_function, batched=True, desc="Tokenizing")
tokenized = tokenized.remove_columns(["text", "idx"])
print(f"Columns: {tokenized['train'].column_names}")

## Step 4: Load Model & Train

In [ ]:
# Load model with 3-class classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=LABEL_MAP,
    label2id={v: k for k, v in LABEL_MAP.items()},
)
print(f"Model parameters: {model.num_parameters():,}")

In [ ]:
# Metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    accuracy = (preds == labels).mean()
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    seed=42,
    report_to="none",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Train!
print("Training started...")
train_result = trainer.train()
print(f"\nTraining complete! Loss: {train_result.training_loss:.4f}")

## Step 5: Evaluate

In [ ]:
eval_results = trainer.evaluate()
print(f"Accuracy:  {eval_results['eval_accuracy']:.4f}")
print(f"Precision: {eval_results['eval_precision']:.4f}")
print(f"Recall:    {eval_results['eval_recall']:.4f}")
print(f"F1 Score:  {eval_results['eval_f1']:.4f}")

## Step 6: Test with Sample Texts

In [ ]:
# Quick test
from transformers import pipeline

pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

test_texts = [
    "I absolutely love this product! It's amazing!",
    "This is the worst experience I've ever had.",
    "The weather is okay today.",
    "I'm so happy and excited about this!",
    "I hate waiting in long lines.",
    "The meeting is at 3pm.",
]

for text in test_texts:
    result = pipe(text)[0]
    print(f"{text:55s} -> {result['label']:10s} ({result['score']:.3f})")

## Step 7: Save & Download Model

In [ ]:
# Save final model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")
print(f"Files: {os.listdir(OUTPUT_DIR)}")

In [ ]:
# Zip the model for download
import shutil
shutil.make_archive("trained_model", "zip", OUTPUT_DIR)
print("Created trained_model.zip")
print(f"Size: {os.path.getsize('trained_model.zip') / 1024 / 1024:.1f} MB")

In [ ]:
# Download the zip file
from google.colab import files
files.download("trained_model.zip")
print("\nDownload started! After downloading:")
print("1. Extract trained_model.zip")
print("2. Copy all files to: Chatbot Project/ml/trained_model/")
print("3. Restart your backend server")